# ボディ検査機 GT2測定値 傾向管理レポート
#### Ver.20261115-01
---

In [ ]:
# ================================================
#  Import
# ================================================
import sys
import sqlite3
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Yu Gothic"
import pandas as pd
from IPython.display import display, Markdown

In [ ]:
# ================================================
# Settings
# ================================================
DB_PATH = Path("../data/body_insp_measurement_data.db")

STD_DATETIME = "2026-11-11 18:06:00"

INSP_MACHINES = [412, 413, 416, 421, 428]
INSP_ITEMS = {
    "total_length": "全長",
    "thickness": "厚さ",
    "bottom_thickness": "ボトム厚",
    "depth_phi071": "深さφ0.71",
    "inner_diameter": "内径",
}

In [ ]:
# Information
print("実行環境:",sys.executable)

print(f"基準日時: {STD_DATETIME}")

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"レポート更新日時: {now}")

In [ ]:

def get_latest_lot_id(insp_machine_no, inspection_item, std_datetime):
    """
    指定した号機・検査項目・基準日時以前の最新測定レコードから
    ロットIDを取得する。

    Returns:
        tuple: (part_no, proc_machine_no, process_date)
        None: 該当レコードがない場合
    """
    sql = """
    SELECT part_no, proc_machine_no, process_date
    FROM measurement_result
    WHERE insp_machine_no = ?
      AND inspection_item = ?
      AND measured_at <= ?
    ORDER BY measured_at DESC
    LIMIT 1
    """

    with sqlite3.connect(DB_PATH) as conn:
        row = conn.execute(
            sql, (insp_machine_no, inspection_item, std_datetime)
        ).fetchone()

    return row


In [ ]:

def get_lot_measurements(insp_machine_no, inspection_item, lot_id):
    """
    指定した検査機番・検査項目・ロットに一致する
    測定データをDataFrameで取得する。

    Args:
        insp_machine_no: 検査機番
        inspection_item: 検査項目
        lot_id: (part_no, proc_machine_no, process_date)

    Returns:
        pd.DataFrame: 測定日時と測定値
    """
    if lot_id is None:
        raise ValueError("ロットIDが指定されていません")

    sql = """
    SELECT *
    FROM measurement_result
    WHERE insp_machine_no = ?
      AND inspection_item = ?
      AND part_no = ?
      AND proc_machine_no = ?
      AND process_date = ?
    ORDER BY measured_at ASC
    """

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(
            sql,
            conn,
            params=(
                insp_machine_no,
                inspection_item,
                *lot_id,
            ),
        )

    return df


In [ ]:
def plot_measurement_trend(df, ax):
    """
    指定したAxesに測定値の折れ線グラフを描画する。

    Args:
        df: 測定データ（DataFrame）
        ax: 描画先のAxes
    """
    if df.empty:
        ax.set_title("No data")
        return

    insp_machine_no = df["insp_machine_no"].iloc[0]

    # 判定結果に応じたマーカー色
    marker_colors = [
        "green" if judgment == "OK" else "red"
        for judgment in df["judgment"]
    ]

    # 測定値の折れ線（黒）
    ax.plot(
        range(len(df)),
        df["measured_value"],
        color="black",
        linewidth=0.6,
    )

    # 測定値のマーカー（OK=緑、NG=赤）
    ax.scatter(
        range(len(df)),
        df["measured_value"],
        c=marker_colors,
        s=9,
        zorder=3,
    )

    # 上限値（赤い点線）
    ax.plot(
        range(len(df)),
        df["upper_limit"],
        color="red",
        linestyle="--",
        linewidth=1,
    )

    # 下限値（赤い点線）
    ax.plot(
        range(len(df)),
        df["lower_limit"],
        color="red",
        linestyle="--",
        linewidth=1,
    )

    # 横軸の左右端に測定日を表示
    start_date = pd.to_datetime(df["measured_at"].iloc[0]).strftime("%m/%d %H:%M")
    end_date = pd.to_datetime(df["measured_at"].iloc[-1]).strftime("%m/%d %H:%M")

    ax.set_xticks([0, len(df) - 1])                  # 横軸の目盛り位置を最初と最後のデータに設定
    ax.set_xticklabels([start_date, end_date])       # 横軸の目盛りラベルを測定日時に設定

    # ロット情報
    part_no = df["part_no"].iloc[0]
    proc_machine_no = df["proc_machine_no"].iloc[0]
    process_date = pd.to_datetime(df["process_date"].iloc[0]).strftime("%Y/%m/%d")

    # 検査結果の集計
    total_count = len(df)
    ok_count = (df["judgment"] == "OK").sum()
    ng_count = (df["judgment"] == "NG").sum()
    ok_ratio = ok_count / total_count * 100
    ng_ratio = ng_count / total_count * 100
    mean_value = df["measured_value"].mean()
    six_sigma = df["measured_value"].std() * 6

    label_text = (
        f"品番={part_no} / 加工機={proc_machine_no} / 加工日={process_date}\n"
        f"検査総数={total_count} / "
        f"OK数={ok_count} ({ok_ratio:.1f}%) / "
        f"NG数={ng_count} ({ng_ratio:.1f}%) / "
        f"平均値={mean_value:.3f} / "
        f"±3σ={six_sigma:.3f}"
    )

    ax.text(
        0, -0.15, label_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
    )
    
    ax.set_title(f"検査機No= {insp_machine_no}", loc="left")
    # ax.set_xlabel("Measurement order")
    # ax.set_ylabel("Measured value")
    ax.grid(True, alpha=0.3)




In [ ]:

def create_measurement_trend_figure(inspection_item, std_datetime):
    """
    指定した検査項目について、5台分の測定トレンドを
    3行×2列に配置したFigureを作成する。

    Returns:
        fig: MatplotlibのFigure
    """
    # 3行×2列のグラフ領域を作成
    fig, axes = plt.subplots(3, 2, figsize=(18, 9))

    # 5台分のグラフを描画
    for insp_machine_no, ax in zip(INSP_MACHINES, axes.flat):

        # 基準日時からロットを取得
        lot_id = get_latest_lot_id(
            insp_machine_no,
            inspection_item,
            std_datetime,
        )

        # ロットが存在しない場合
        if lot_id is None:
            ax.set_title(f"検査機No=  {insp_machine_no}", loc="left")
            ax.text(
                0.5, 0.5, "No data",
                ha="center", va="center",
                transform=ax.transAxes,
            )
            ax.grid(True, alpha=0.3)
            continue

        # ロットの測定データを取得
        df = get_lot_measurements(
            insp_machine_no,
            inspection_item,
            lot_id,
        )

        # 1台分のグラフを描画
        plot_measurement_trend(df, ax)

    # 余った6番目のAxesを非表示
    axes.flat[5].set_visible(False)

    fig.suptitle(
    f"測定項目={INSP_ITEMS[inspection_item]}",
        fontsize=18,
        x=0.02,
        ha="left",
    )
    
    fig.tight_layout()

    return fig


In [ ]:
# 検査項目ごとにFigureを作成・表示
for inspection_item in INSP_ITEMS:
    # Figure間の区切り線
    display(Markdown("---"))

    fig = create_measurement_trend_figure(
        inspection_item,
        STD_DATETIME,
    )

    fig.subplots_adjust(hspace=0.65)

    plt.show()
